In [1]:
import torch

In [2]:
class simpleLSTM(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.linear = torch.nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, state = None):
        batch_size = x.size(0)

        if state is None:
            h_prev = torch.zeros(batch_size, self.hidden_size)
            c_prev = torch.zeros(batch_size, self.hidden_size)
        else:
            h_prev, c_prev = state

        combined = torch.cat([x, h_prev], dim=1)

        gates = self.linear(combined)

        i, f, c_cap, o = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        c_cap = torch.tanh(c_cap)
        o = torch.sigmoid(o)

        c = f * c_prev + i * c_cap

        h = o * torch.tanh(c)

        return h, c

In [3]:
class multiLayerLSTM(torch.nn.Module):
    def __init__(self, input_size: int, hidden_size: int, layers: int = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.layers = layers

        self.cells = torch.nn.ModuleList()

        for layer in range(layers):
            current_input_size = input_size if layer == 0 else hidden_size
            self.cells.append(torch.nn.Linear(current_input_size + hidden_size, 4 * hidden_size))

    def forward(self, x, states = None):
        batch_size, seq_len, _ = x.shape

        if states is None:
            states = [
                (torch.zeros(batch_size, self.hidden_size, device = x.device),
                torch.zeros(batch_size, self.hidden_size, device = x.device))
                for _ in range(self.layers)
            ]

        sequence_outputs = []

        for t in range(seq_len):
            layer_input = x[:, t, : ]

            for layer in range(self.layers):

                h_prev, c_prev = states[layer]

                combined = torch.cat([layer_input, h_prev], dim=1)

                gates = self.cells[layer](combined)

                i, f, c_cap, o = gates.chunk(4, dim = 1)

                c = torch.sigmoid(f) * c_prev + torch.sigmoid(i) * torch.tanh(c_cap)
                h = torch.sigmoid(o) * torch.tanh(c)

                states[layer] = (h, c)

                layer_input = h

            sequence_outputs.append(h)

        out_tensor = torch.stack(sequence_outputs, dim=1)

        return out_tensor, states

In [4]:
class TextGenerator(torch.nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_size)
        
        self.lstm = multiLayerLSTM(embed_size, hidden_size, num_layers)
        
        self.fc = torch.nn.Linear(hidden_size, vocab_size)

    def forward(self, x, states=None):
        
        embedded = self.embedding(x) 
        
        lstm_out, states = self.lstm(embedded, states)
        
        logits = self.fc(lstm_out.reshape(-1, lstm_out.size(2)))
        
        return logits, states

In [2]:
from torch.utils.data import Dataset, DataLoader

In [3]:
class TextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [4]:
import urllib.request

In [5]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode('utf-8')

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {c: i for i, c in enumerate(chars)}

encoded_text = [char_to_int[c] for c in text]

In [6]:
split_idx = int(len(encoded_text) * 0.8)
train_data = encoded_text[:split_idx]
test_data = encoded_text[split_idx:]

seq_len = 50
batch_size = 128

train_dataset = TextDataset(train_data, seq_len)
test_dataset = TextDataset(test_data, seq_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
model = TextGenerator(vocab_size=vocab_size, embed_size=32, hidden_size=128, num_layers = 3)
model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [12]:
epochs = 20

print(f"Vocabulary Size: {vocab_size}")
print(f"Training Batches: {len(train_loader)} | Testing Batches: {len(test_loader)}\n")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        logits, _ = model(x, states=None) 
        
        loss = criterion(logits, y.view(-1))
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    model.eval()
    test_loss = 0.0
    
    with torch.inference_mode():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            
            logits, _ = model(x, states=None)
            loss = criterion(logits, y.view(-1))
            
            test_loss += loss.item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

Vocabulary Size: 61
Training Batches: 624 | Testing Batches: 155

Epoch 01/20 | Train Loss: 2.1201 | Test Loss: 1.6989
Epoch 02/20 | Train Loss: 1.4412 | Test Loss: 1.6525
Epoch 03/20 | Train Loss: 1.1973 | Test Loss: 1.7717
Epoch 04/20 | Train Loss: 1.0095 | Test Loss: 1.9627
Epoch 05/20 | Train Loss: 0.8672 | Test Loss: 2.1530
Epoch 06/20 | Train Loss: 0.7568 | Test Loss: 2.3593
Epoch 07/20 | Train Loss: 0.6741 | Test Loss: 2.5674
Epoch 08/20 | Train Loss: 0.6102 | Test Loss: 2.7168
Epoch 09/20 | Train Loss: 0.5607 | Test Loss: 2.8836
Epoch 10/20 | Train Loss: 0.5245 | Test Loss: 2.9577
Epoch 11/20 | Train Loss: 0.4934 | Test Loss: 3.0909


KeyboardInterrupt: 

In [9]:
class improvedLSTM(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_size: int, hidden_size: int, num_layers: int = 1, dropout: float = 0.2):
        super().__init__()

        self.hidden_size = hidden_size
        self.layers = num_layers
        self.vocab_size = vocab_size
        self.embedding_size = embedding_size

        self.dropout = torch.nn.Dropout(p=dropout)
        self.embedding = torch.nn.Embedding(vocab_size, embedding_size)
        self.cells = torch.nn.ModuleList()

        for layer in range(self.layers):
            current_input_size = self.embedding_size if layer == 0 else hidden_size
            self.cells.append(torch.nn.Linear(current_input_size + hidden_size, 4 * hidden_size))

        self.classifier = torch.nn.Sequential(
            self.dropout,
            torch.nn.Linear(hidden_size, hidden_size * 2), 
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_size * 2, vocab_size) 
        )

    def forward(self, x, states=None):
        embedded = self.embedding(x) 
        
        batch_size, seq_len, _ = embedded.shape

        if states is None:
            states = [
                (torch.zeros(batch_size, self.hidden_size, device=embedded.device),
                 torch.zeros(batch_size, self.hidden_size, device=embedded.device))
                for _ in range(self.layers)
            ]

        sequence_outputs = []

        for t in range(seq_len):
            layer_input = embedded[:, t, :]

            for layer in range(self.layers):
                h_prev, c_prev = states[layer]
                combined = torch.cat([layer_input, h_prev], dim=1)
                gates = self.cells[layer](combined)
                i, f, c_cap, o = gates.chunk(4, dim=1)

                c = torch.sigmoid(f) * c_prev + torch.sigmoid(i) * torch.tanh(c_cap)
                h = torch.sigmoid(o) * torch.tanh(c)

                states[layer] = (h, c)

                if layer < self.layers - 1:
                    layer_input = self.dropout(h)
                else:
                    layer_input = h

            sequence_outputs.append(h)

        out_tensor = torch.stack(sequence_outputs, dim=1)

        logits = self.classifier(out_tensor)

        return logits, states

In [24]:
narrow_model = improvedLSTM(vocab_size=vocab_size, embedding_size=32, hidden_size=64, num_layers=3, dropout=0.2 )
narrow_model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(narrow_model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [27]:
epochs = 20

for epoch in range(epochs):
    narrow_model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
                
        optimizer.zero_grad(set_to_none=True)

        logits, _ = narrow_model(x)

        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(narrow_model.parameters(),max_norm=1.0)

        optimizer.step()
                
        train_loss += loss.item()
        
    narrow_model.eval()
    test_loss = 0.0
    with torch.inference_mode():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
                    
            logits, _ = narrow_model(x, states=None)
            loss = criterion(logits.view(-1, vocab_size), y.view(-1))
                    
            test_loss += loss.item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1:02d} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")
    
    scheduler.step(avg_test_loss)

Epoch 01 | LR: 0.001000 | Train Loss: 1.8217 | Test Loss: 1.8068
Epoch 02 | LR: 0.001000 | Train Loss: 1.6378 | Test Loss: 1.7671
Epoch 03 | LR: 0.001000 | Train Loss: 1.5978 | Test Loss: 1.7593
Epoch 04 | LR: 0.001000 | Train Loss: 1.5776 | Test Loss: 1.7471
Epoch 05 | LR: 0.001000 | Train Loss: 1.5646 | Test Loss: 1.7411
Epoch 06 | LR: 0.001000 | Train Loss: 1.5551 | Test Loss: 1.7405
Epoch 07 | LR: 0.001000 | Train Loss: 1.5481 | Test Loss: 1.7344
Epoch 08 | LR: 0.001000 | Train Loss: 1.5423 | Test Loss: 1.7326
Epoch 09 | LR: 0.001000 | Train Loss: 1.5376 | Test Loss: 1.7281
Epoch 10 | LR: 0.001000 | Train Loss: 1.5338 | Test Loss: 1.7228
Epoch 11 | LR: 0.001000 | Train Loss: 1.5304 | Test Loss: 1.7217
Epoch 12 | LR: 0.001000 | Train Loss: 1.5274 | Test Loss: 1.7188
Epoch 13 | LR: 0.001000 | Train Loss: 1.5248 | Test Loss: 1.7198
Epoch 14 | LR: 0.001000 | Train Loss: 1.5224 | Test Loss: 1.7193
Epoch 15 | LR: 0.001000 | Train Loss: 1.5202 | Test Loss: 1.7259
Epoch 16 | LR: 0.000500 |

In [28]:
int_to_char = {i: c for i, c in enumerate(chars)}

In [29]:
def generate_text(model, start_string, generation_length, temperature=1.0):
    model.eval()
    
    input_seq = [char_to_int[char] for char in start_string]
    x = torch.tensor([input_seq], dtype=torch.long).to(device)
    
    generated_text = start_string
    states = None 
    
    with torch.inference_mode():
        for _ in range(generation_length):
            logits, states = model(x, states)
            
            next_char_logits = logits[0, -1, :] 
            
            next_char_logits = next_char_logits / temperature
            
            probs = torch.softmax(next_char_logits, dim=0)
            
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            
            generated_text += int_to_char[next_char_idx]
            
            x = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
            
    return generated_text

In [30]:
print(generate_text(narrow_model, start_string="O Romeo, ", generation_length=300, temperature=0.8))

O Romeo, we word hath beat begun's from the
bodyster till they repain to me;
Or it is tell thee that, for his death?

KING HENRY VI:
I am o'erseeds with in that Remand of Rome: he bear my life,
It she say I'll so my necessate,
And with your fearful stronger deliverion.

DUKE OF AUMERLE:
Be shall we thank the


In [31]:
print(generate_text(narrow_model, start_string="O Romeo, ", generation_length=300, temperature=0.4))

O Romeo, I will say the stroke to promised to me
That thou shalt prince my grace was to such to hear the hands,
That you have seen the country to heaven, sir, sir, that shall death the death,
That he shall prove the first common that we are unto the children,
And the presently with with their side should hav


In [32]:
from pathlib import Path

In [33]:
save_path = Path('models/shakespeare_lstm_64_weights.pth')
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(narrow_model, save_path)
print(f"Model successfully saved to {save_path}")

Model successfully saved to models\shakespeare_lstm_64_weights.pth
